In [ ]:
!pip install -q huggingface_hub peft transformers==4.46.0 accelerate torch bitsandbytes

In [ ]:
from huggingface_hub import login

# Va sur https://huggingface.co/settings/tokens
# Crée un token avec permission "Write"
login()

In [ ]:
# Upload ton zip modèle
from google.colab import files
import zipfile, os

print("Upload ton fichier whisper-wolof-model-p2.zip...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
os.makedirs('/content/model', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/model')

print("Modèle extrait !")
print(os.listdir('/content/model'))

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

MODEL_ID = "openai/whisper-large-v3"
ADAPTER_PATH = "/content/model"

# ===== CHANGE CE NOM PAR TON USERNAME HF =====
HF_REPO = "TON-USERNAME/whisper-wolof-v1"
# =============================================

print("Chargement du processeur...")
processor = WhisperProcessor.from_pretrained(ADAPTER_PATH)

print("Chargement de Whisper Large V3 (fp16)...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Application et fusion du LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.merge_and_unload()

print(f"\nPush vers HuggingFace: {HF_REPO}...")
print("(~3 Go à upload, ça prend 5-10 min)")
model.push_to_hub(HF_REPO, safe_serialization=True)
processor.push_to_hub(HF_REPO)

print(f"\n{'='*60}")
print(f"   MODÈLE PUBLIÉ AVEC SUCCÈS !")
print(f"   https://huggingface.co/{HF_REPO}")
print(f"")
print(f"   L'Inference API se lance automatiquement.")
print(f"   Ton modèle est maintenant disponible 24/7 !")
print(f"{'='*60}")